In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/weather_daily.csv", parse_dates=["time"])
df = df.sort_values(["city", "time"]).reset_index(drop=True)

In [4]:
g = df.groupby("city")

# Rain building up over recent days (waterlogged ground, flooding risk)
df["rain_3d"] = g["precipitation_sum"].transform(lambda s: s.rolling(3, min_periods=1).sum())
df["rain_7d"] = g["precipitation_sum"].transform(lambda s: s.rolling(7, min_periods=1).sum())

# How unusual today's rain is FOR THIS CITY (20 mm means more in London than Glasgow)
city_p95 = g["precipitation_sum"].transform(lambda s: s.quantile(0.95))
df["heavy_rain_day"] = (df["precipitation_sum"] >= city_p95).astype(int)

# Simple extreme-weather flags
df["freezing"] = (df["temperature_2m_min"] < 0).astype(int)
df["hot_day"] = (df["temperature_2m_max"] >= 28).astype(int)   # heat-related speed restrictions
df["storm_gusts"] = (df["wind_gusts_10m_max"] >= 75).astype(int)
df["any_snow"] = (df["snowfall_sum"] > 0).astype(int)

# Calendar features
df["month"] = df["time"].dt.month
df["day_of_week"] = df["time"].dt.day_of_week

df.head()

,city,time,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,wind_gusts_10m_max,rain_3d,rain_7d,heavy_rain_day,freezing,hot_day,storm_gusts,any_snow,month,day_of_week
0,Birmingham,2023-01-01,9.6,6.3,1.4,1.4,0.0,29.2,54.7,1.4,1.4,0,0,0,0,0,1,6
1,Birmingham,2023-01-02,6.3,0.8,0.0,0.0,0.0,15.2,27.4,1.4,1.4,0,0,0,0,0,1,0
2,Birmingham,2023-01-03,11.5,1.2,2.2,2.2,0.0,32.8,60.8,3.6,3.6,0,0,0,0,0,1,1
3,Birmingham,2023-01-04,12.1,9.3,2.0,2.0,0.0,33.1,64.4,4.2,5.6,0,0,0,0,0,1,2
4,Birmingham,2023-01-05,11.7,7.9,0.7,0.7,0.0,33.0,61.2,4.9,6.3,0,0,0,0,0,1,3


In [5]:
df.groupby("city")[["heavy_rain_day", "freezing", "hot_day", "storm_gusts", "any_snow"]].sum()

,heavy_rain_day,freezing,hot_day,storm_gusts,any_snow
city,,,,,
Birmingham,55,90,11,29,31
Bristol,55,68,10,43,23
Edinburgh,55,77,1,70,63
Glasgow,55,129,1,24,62
Leeds,55,93,5,29,32
London,56,69,29,14,22
Manchester,55,89,7,15,46
York,55,94,5,15,30
